[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/optimization/kes/currency-MV.ipynb)

In [ ]:
if 'google.colab' in str(get_ipython()):
    import os
    os.system('pip -qqq install ModelFlowIb openpyxl')
    os.system(
        'curl -sL -o exchangerates_get.py '
        'https://raw.githubusercontent.com/IbHansen/wb-debt-simulation/main/'
        'optimization/exchangerates_get.py'
    )
    os.system(
        'curl -sL -o get_kes.py '
        'https://raw.githubusercontent.com/IbHansen/wb-debt-simulation/main/'
        'optimization/kes/get_kes.py'
    )
    os.makedirs('currency', exist_ok=True)
    print("Please upload 'kes_wide.xlsx' when the file picker appears:")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded:
        import shutil; shutil.move(fn, f'currency/{fn}')


In [ ]:
import sys
from pathlib import Path
# Make exchangerates_get (in the parent folder) importable when running locally
if str(Path('..').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import exchangerates_get as er
from get_kes import get_kes_iso


In [ ]:
# Load KES FX rates and convert to KES-as-base convention.
# get_kes_iso() reads currency/kes_wide.xlsx, inverts CBK rates
# (USD_KES -> KES_USD etc.) and resamples to quarterly frequency --
# identical pipeline to ecb_fx_eur + convert_base_currency(base='kes').
fx_ccy = get_kes_iso(
    currencies=['USD', 'GBP', 'EUR', 'JPY', 'CNY'],
    start='2016-01-01',
    freq='QE',
    agg='last',
)

fx_returns = er.get_fx_returns(fx_ccy)

# Annualised covariance: quarterly cov x 4
fx_cov = er.get_fx_covariance(fx_returns, periods_per_year=4)


In [ ]:
# Strip 'KES_' prefix so the optimiser index is just the currency code
cov_df = fx_cov.rename(
    index=lambda x: x.split('_')[1],
    columns=lambda x: x.split('_')[1],
)
names = cov_df.index
n = len(names)

# Approximate sovereign borrowing costs (annual) from a KES perspective.
# Adjust to reflect current market conditions or your own assumptions.
interest_rates = {
    'USD': 0.043,
    'GBP': 0.045,
    'EUR': 0.025,
    'JPY': 0.005,
    'CNY': 0.025,
}

assumptions = pd.DataFrame(
    {
        'interest_rate':         [interest_rates.get(c, 0.03) for c in names],
        'expected_appreciation': [0.0] * n,
        'min_share':             [0.0] * n,
        'max_share':             [1.0] * n,
        'current_share':         [1.0 / n] * n,
    },
    index=names,
)
assumptions


## Interactive inputs -- `DebtFrontierInputs`

Edit any cell and press **Run frontier** to re-solve and re-plot.

In [ ]:
inputs = er.DebtFrontierInputs(cov_df=cov_df, assumptions=assumptions)
inputs.assumptions

In [ ]:
inputs.widget()

## Additional frontier -- including domestic (KES) debt

Domestic KES debt carries **zero FX risk** (covariance with all foreign
currencies is zero) but typically a higher nominal interest rate. Adding it
extends the frontier to lower-risk portfolios.

The default domestic rate below is 12 % -- adjust in the widget.

In [ ]:
# Extend covariance matrix with a KES (domestic) row/column of zeros -- no FX risk.
cov_df_dom = cov_df.reindex(
    index=list(cov_df.index) + ['KES'],
    columns=list(cov_df.columns) + ['KES'],
    fill_value=0.0,
)

# KES assumption row -- current_share=0 keeps foreign sum=1 in the widget indicator
kes_row = pd.DataFrame(
    {
        'interest_rate':         [0.12],   # ~12 % domestic rate -- placeholder
        'expected_appreciation': [0.0],
        'min_share':             [0.0],
        'max_share':             [1.0],
        'current_share':         [0.0],
    },
    index=['KES'],
)
assumptions_dom = pd.concat([assumptions, kes_row])

inputs_dom = er.DebtFrontierInputs(
    cov_df=cov_df_dom,
    assumptions=assumptions_dom,
    name='kes_basis_with_domestic',
    chartfolder='graph/',
)
inputs_dom.widget()


## Scripted scenario loop (no widget)

For batch runs or report generation, use `inputs.plot()` directly.

In [ ]:
scenarios = {
    'dom_10pct': 0.10,
    'dom_12pct': 0.12,
    'dom_15pct': 0.15,
}

for scenario_name, dom_rate in scenarios.items():
    inp = er.DebtFrontierInputs(
        cov_df=cov_df_dom,
        assumptions=assumptions_dom.copy(),
        name=scenario_name,
        chartfolder='graph/',
    )
    inp.assumptions.at['KES', 'interest_rate'] = dom_rate
    print(f'Running scenario: {scenario_name}  (KES rate = {dom_rate:.0%})')
    inp.plot(export_formats=('svg',))
